# E06: can simple checks make cheap AI reliable?

Development pilot, 25 October 2026. Reproduce saved-result analysis without API calls or keys.
The revised pilot has 21 items per model, one repeat; it cannot measure three-run agreement.
The original pilot is preserved and Sol has only invoice coverage.


In [ ]:
from pathlib import Path
import json
import subprocess
import sys

root = Path.cwd()
if not (root / "analyse.py").exists():
    root = root / "e06-cheap-model-good-enough"
assert (root / "analyse.py").exists(), "Open this notebook from the experiment folder or repository root"

def execute(*args):
    result = subprocess.run([sys.executable, *args], cwd=root, text=True, capture_output=True)
    print(result.stdout)
    if result.returncode:
        print(result.stderr)
    result.check_returncode()


## Revised pilot and original checks
Both validators score the same saved model answers. The revised validator was developed after inspecting pilot data.


In [ ]:
execute("analyse.py", "--items", "items_samples.jsonl", "--repeats", "1", "--run-tag", "v2")
execute("analyse.py", "--items", "items_samples.jsonl", "--repeats", "1", "--run-tag", "v2", "--checks", "baseline")
summary = json.loads((root / "analysis_samples_v2/summary-v2.json").read_text())
for model, tasks in summary["models"].items():
    print(model, {task: f"{round(data['accuracy'] * data['n'])}/{data['n']}" for task, data in tasks.items()})
    print("Revised checks:", summary["checks_by_version"][model]["v2"]["all"])


## Original pilot: incomplete fallback coverage
No overall cascade estimate is produced from Sol's invoice-only original run.


In [ ]:
execute("analyse.py", "--items", "items_samples.jsonl", "--repeats", "1")


## Inspect every incorrect answer
This selection rule includes all mistakes, not just the dramatic ones. The item key is unchanged.


In [ ]:
items = {r["id"]: r for r in map(json.loads, (root / "items_samples.jsonl").read_text().splitlines())}
audit = [json.loads(line) for line in (root / "analysis_samples_v2/row-audit.jsonl").read_text().splitlines()]
for row in audit:
    if row["correct"]:
        continue
    raw = json.loads((root / "results_samples_v2" / row["file"]).read_text().splitlines()[row["line"] - 1])
    print(row["model"], row["item"], row["errors"], "checks passed:", row["v2_check_passed"])
    print("Expected:", items[row["item"]]["gold"])
    print("Actual:", raw["tool_calls"] or raw["content"])


## Reproduce charts
Requires matplotlib: install `requirements-dev.txt` in your Python environment first. The notebook does not install packages or call models.


In [ ]:
execute("make_figures.py", "--summary", "analysis_samples_v2/summary-v2.json")
from IPython.display import SVG, display
for path in sorted((root / "analysis_samples_v2/figures").glob("*.svg")):
    display(SVG(filename=str(path)))


## Regression checks and limits
No API calls. The unseen set, independent key audit and item-level uncertainty analysis remain pending; see `PROTOCOL.md` and `PUBLICATION.md`. Costs use configured list prices and omit unknown failed-attempt charges.


In [ ]:
execute("-m", "unittest", "-v", "test_e06.py")
